# Task 1: Google Play Store Review Scraper & Preprocessing
## 10 Academy Week 2 Challenge: Fintech Customer Experience Analytics

This notebook implements the data engineering pipeline to scrape, clean, and preprocess customer reviews for the three major Ethiopian banking apps:
1. **CBE (Commercial Bank of Ethiopia)** - `prod.cbe.birr`
2. **BOA (Bank of Abyssinia - Apollo)** - `com.boa.apollo`
3. **Dashen Bank (Amole)** - `com.cr2.amolelight`

## Setup & Libraries

In [1]:
import pandas as pd
from google_play_scraper import Sort, reviews
from datetime import datetime
import os
import sys

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Define Ingestion & Scraping Logic
We scrape up to 600 reviews per bank to ensure a statistically robust baseline, focusing on recent user feedback.

In [2]:
def scrape_bank_reviews(app_id, bank_name, num_reviews=600):
    print(f"Scraping reviews for {bank_name} ({app_id})...")
    result = []
    count = 0
    
    # Fetch reviews in batches
    try:
        batch, token = reviews(
            app_id,
            lang='en',
            country='us', # US/Global country code contains most English text
            sort=Sort.NEWEST,
            count=num_reviews
        )
        result.extend(batch)
    except Exception as e:
        print(f"Error scraping {bank_name}: {e}")
        return pd.DataFrame()

    # Extract required fields
    columns = ['reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'at', 'appVersion']
    data = []
    for r in result:
        row = {
            'id': r.get('reviewId'),
            'review': r.get('content'),
            'rating': r.get('score'),
            'date': r.get('at'),
            'bank': bank_name,
            'source': 'Google Play Store'
        }
        data.append(row)
        
    df = pd.DataFrame(data)
    print(f"Successfully collected {len(df)} raw reviews for {bank_name}.")
    return df

## Execute Data Extraction Pipeline

In [3]:
app_ids = {
    'CBE': 'prod.cbe.birr',
    'BOA': 'com.boa.apollo',
    'Dashen': 'com.cr2.amolelight'
}

dfs = []
for bank, app_id in app_ids.items():
    df_bank = scrape_bank_reviews(app_id, bank, 600)
    if not df_bank.empty:
        dfs.append(df_bank)

raw_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"Total combined raw reviews: {len(raw_df)}")

Scraping reviews for CBE (prod.cbe.birr)...


Successfully collected 600 raw reviews for CBE.
Scraping reviews for BOA (com.boa.apollo)...


Successfully collected 600 raw reviews for BOA.
Scraping reviews for Dashen (com.cr2.amolelight)...


Successfully collected 505 raw reviews for Dashen.
Total combined raw reviews: 1705


## Data Cleaning & Preprocessing Steps
1. **De-duplication**: Filter unique reviews using `id`.
2. **Null Filtering**: Drop records with missing text/rating.
3. **Date Standardization**: Normalize dates to `YYYY-MM-DD` ISO format.

In [4]:
def preprocess_data(df):
    if df.empty:
        return df
    
    # 1. Remove duplicates
    initial_count = len(df)
    df = df.drop_duplicates(subset=['id'])
    print(f"Removed {initial_count - len(df)} duplicates.")
    
    # 2. Filter null reviews/ratings
    df = df.dropna(subset=['review', 'rating'])
    
    # 3. Format Date to YYYY-MM-DD
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    
    return df

cleaned_df = preprocess_data(raw_df)
print(f"Total processed and cleaned reviews: {len(cleaned_df)}")

Removed 0 duplicates.
Total processed and cleaned reviews: 1705


## Save Cleaned Dataset
We serialize the resulting dataset into `data/raw/cleaned_reviews.csv`.

In [5]:
os.makedirs('../data/raw', exist_ok=True)
cleaned_df.to_csv('../data/raw/cleaned_reviews.csv', index=False)
print("Cleaned reviews successfully saved to '../data/raw/cleaned_reviews.csv'.")

Cleaned reviews successfully saved to '../data/raw/cleaned_reviews.csv'.


## Exploratory Data Summary

In [6]:
print(cleaned_df.groupby('bank')['rating'].describe())
print("\nFirst 5 cleaned records:")
print(cleaned_df.head())

        count      mean       std  min  25%  50%  75%  max
bank                                                      
BOA     600.0  3.406667  1.807411  1.0  1.0  5.0  5.0  5.0
CBE     600.0  4.235000  1.391523  1.0  4.0  5.0  5.0  5.0
Dashen  505.0  4.093069  1.449292  1.0  4.0  5.0  5.0  5.0

First 5 cleaned records:
                                     id  \
0  aa922d75-c4b1-4df0-86ba-1a87397f6104   
1  f75273d1-e02c-45b7-b399-6bfae8ae4956   
2  7bc6523d-d7df-41fa-8af4-cdc2de34efa2   
3  7d23d03e-3ffc-47ce-83aa-77ce2c009417   
4  289e3376-e37b-4e1e-93fa-3613b53ca84a   

                                        review  rating        date bank  \
0              Whynot be fast?and be open fast       5  2026-05-16  CBE   
1                                    excellent       4  2026-05-16  CBE   
2         CBE One of the best in Ethiopia bink       4  2026-05-15  CBE   
3                                       good 👍       5  2026-05-13  CBE   
4  fast & safe banking changes your carrier .